Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_movielens
from src.content_based import ContentBasedRecommender
from src.popularity import PopularityRecommender
from src.hybrid import HybridRecommender
from src.random_baseline import RandomRecommender
from src.evaluation import train_test_split_by_time, compare_models

Load Dataset

In [ ]:
ratings, movies, users = load_movielens()

Initialize the Model

In [ ]:
model = ContentBasedRecommender(movies)

Generate Recommendations

In [ ]:
recommendations = model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(recommendations)

## Popularity-Based Recommendations

The baseline recommends movies with at least `min_ratings` ratings and the highest share of positive (>= threshold) ratings. Every user gets the same list, minus movies they've already rated.

In [ ]:
popularity_model = PopularityRecommender(movies, min_ratings=50)

popular_recommendations = popularity_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(popular_recommendations)

## Hybrid Recommendations

Combines both scores: `score = alpha * content_score + (1 - alpha) * popularity_score`. Both scores are rescaled to 0-1 first. `alpha=1` is pure content-based and `alpha=0` is pure popularity.

In [ ]:
hybrid_model = HybridRecommender(movies, alpha=0.5, min_ratings=50)

hybrid_recommendations = hybrid_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(hybrid_recommendations)

## Evaluation

Each user's most recent 20% of ratings are held out as a test set. Each model only sees the training ratings, and we check how many of its top-10 recommendations are movies the user went on to rate 4 or higher.

- **precision@10**: share of the 10 recommendations the user liked
- **recall@10**: share of the user's liked held-out movies that were recommended
- **hit_rate@10**: share of users with at least one liked movie in their top 10
- **coverage**: share of the whole catalog recommended to at least one user

The random recommender is a floor: any useful model should clearly beat it on precision, recall and hit rate (random will always win on coverage).

In [ ]:
train, test = train_test_split_by_time(ratings, test_fraction=0.2)

print(f"Train ratings: {len(train):,}")
print(f"Test ratings:  {len(test):,}")

In [ ]:
models = {
    "Random (floor)": RandomRecommender(movies, seed=0),
    "Content-based": ContentBasedRecommender(movies),
    "Popularity": PopularityRecommender(movies, min_ratings=50),
    "Hybrid (alpha=0.5)": HybridRecommender(movies, alpha=0.5, min_ratings=50),
}

comparison = compare_models(models, train, test, k=10, threshold=4)
display(comparison.round(4))

In [ ]:
metrics = ["precision@10", "recall@10", "hit_rate@10", "coverage"]

fig, axes = plt.subplots(1, len(metrics), figsize=(14, 4))
for ax, metric in zip(axes, metrics):
    comparison[metric].plot.bar(ax=ax)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Recommender comparison")
plt.tight_layout()
plt.show()

### Choosing alpha

Run the hybrid with different weights to see how shifting from popularity (left) to genre similarity (right) changes accuracy and coverage. This takes a minute or two.

In [ ]:
alphas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

sweep = compare_models(
    {a: HybridRecommender(movies, alpha=a, min_ratings=50) for a in alphas},
    train, test, k=10, threshold=4
)
sweep.index.name = "alpha"
display(sweep.round(4))

In [ ]:
fig, axes = plt.subplots(1, len(metrics), figsize=(14, 4))
for ax, metric in zip(axes, metrics):
    ax.plot(sweep.index, sweep[metric], marker="o")
    ax.set_title(metric)
    ax.set_xlabel("alpha")

plt.suptitle("Hybrid performance by alpha (0 = popularity, 1 = content-based)")
plt.tight_layout()
plt.show()

In [ ]:
best_alpha = sweep["precision@10"].idxmax()
print(f"Best alpha by precision@10: {best_alpha}")
print(f"Hybrid at best alpha:  {sweep.loc[best_alpha, 'precision@10']:.4f}")
print(f"Content-based alone:   {sweep.loc[1.0, 'precision@10']:.4f}")
print(f"Popularity alone:      {sweep.loc[0.0, 'precision@10']:.4f}")

### Robustness check

The comparison above uses one split. Re-run it with different hold-out sizes to check the ranking of the models doesn't depend on that choice. This also takes a minute or two.

In [ ]:
robustness = {}
for fraction in [0.1, 0.2, 0.3]:
    tr, te = train_test_split_by_time(ratings, test_fraction=fraction)
    robustness[fraction] = compare_models(
        {
            "Random (floor)": RandomRecommender(movies, seed=0),
            "Content-based": ContentBasedRecommender(movies),
            "Popularity": PopularityRecommender(movies, min_ratings=50),
            f"Hybrid (alpha={best_alpha})": HybridRecommender(movies, alpha=best_alpha, min_ratings=50),
        },
        tr, te, k=10, threshold=4
    )["precision@10"]

robustness = pd.DataFrame(robustness)
robustness.columns.name = "test_fraction"
display(robustness.round(4))

### Results

Fill these in from the tables and charts above (see the README section "Writing up results").

**Model comparison.** [Which model had the highest precision@10, recall@10 and hit_rate@10? Give the numbers. How far above the random floor was each model?]

**Coverage.** [Which model recommended the widest share of the catalog, and which the narrowest? What does that trade-off mean for users?]

**Choosing alpha.** [Which alpha gave the best precision@10? Describe how the metrics changed as alpha went from 0 (popularity) to 1 (content-based).]

**Robustness.** [Did the ranking of the models stay the same for test_fraction 0.1, 0.2 and 0.3?]

**Hypothesis.** [We hypothesized personalized recommendations would beat popularity, and the hybrid would balance the two. Was that supported? If content-based did worse than popularity, a likely reason is that genre is a coarse signal: many movies share the same genres.]